[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-9_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 93. The Inventory-Routing Problem (IRP)
## Tier 9: A QUBO and QAOA modeling study

### Purpose and scope

This notebook demonstrates how a simplified inventory-routing model can be represented as a Quadratic Unconstrained Binary Optimization (QUBO) objective and how that objective can be connected to a Quantum Approximate Optimization Algorithm (QAOA) circuit design. The example uses one depot, eight customers, two vehicles, and four planning periods. It constructs binary decisions, adds penalty terms for selected constraints, and reports the resulting model and sampled candidate solutions.

The notebook does **not** run on quantum hardware or implement a quantum state-vector simulator. Its sampling routine is an explicitly labeled classical surrogate for illustrating the surrounding workflow; it does not execute the designed QAOA circuit, estimate a QAOA expectation value, or establish quantum advantage. The circuit object is a schematic gate list, not a hardware-ready circuit.

### Modeling assumptions

- Customer locations are two-dimensional, and the depot is at the origin. The distance term is the straight-line distance from the depot to a customer; it does not represent the order of stops or distance between successive customers.
- A route decision means that a vehicle visits a customer in a period. The model does not determine visit order.
- Delivery and inventory are binary proxies. The delivery variable is linked to the number of visits, and the inventory proxy is linked to delivery in the same period. This is not a continuous inventory-balance model: it omits demand evolution, replenishment quantities, depot inventory flow, and safety-stock dynamics.
- Each customer can be visited by at most one vehicle in a period. A vehicle can visit at most three customers per period. Binary slack variables encode this capacity limit.
- Penalty weights are modeling parameters. A finite penalty does not automatically guarantee that every lowest-energy bit string is feasible; feasibility must be checked on the sampled candidates.

### QUBO formulation

Let $x_{vct}\in\{0,1\}$ indicate whether vehicle $v$ visits customer $c$ in period $t$, $y_{ct}\in\{0,1\}$ indicate a delivery, $z_{ct}\in\{0,1\}$ denote the inventory proxy, and $a_{vt}\in\{0,1\}$ indicate vehicle use. The model also uses binary slack bits $s_{vt1}$ and $s_{vt2}$ for the capacity constraint. The QUBO is

$$
E(q)=C(q)+P\sum_{c,t}\left(y_{ct}-\sum_v x_{vct}\right)^2
+P\sum_{c,t}(z_{ct}-y_{ct})^2
+P\sum_{v,c,t}x_{vct}(1-a_{vt})
+P\sum_{v,t}\left(\sum_c x_{vct}+s_{vt1}+2s_{vt2}-3\right)^2
+P\sum_{c,t}\sum_{v<w}x_{vct}x_{wct},
$$

where the modeled operating cost is

$$
C(q)=\sum_{v,c,t}d_c\,r_v x_{vct}
+\sum_{v,t}f_v a_{vt}
+0.5\sum_{c,t}y_{ct}
+\sum_{c,t}h_c z_{ct}.
$$

Here, $d_c$ is the depot-to-customer Euclidean distance, $r_v$ is vehicle $v$'s variable cost per distance unit, $f_v$ is its fixed cost per used period, $h_c$ is customer $c$'s holding-cost coefficient, and $P$ is the penalty weight. The capacity equation uses two slack bits, which represent values from zero through three; together with the route variables, the equality enforces no more than three visits. The pairwise visit penalty prevents two vehicles from visiting the same customer in the same period.

The implementation stores a QUBO as an upper-triangular matrix $Q$ and evaluates $E(q)=\sum_i Q_{ii}q_i+\sum_{i<j}Q_{ij}q_iq_j$. A squared penalty is expanded algebraically before being added to that matrix. The code's `qubo_matrix` and `_calculate_energy` implement this convention.

### QAOA context and workflow

QAOA alternates a problem unitary, derived from the cost Hamiltonian, with a mixer unitary that explores binary assignments. At depth $p$, the circuit has $p$ pairs of parameters $(\gamma_k,\beta_k)$, for $2p$ parameters in total. A classical optimizer normally updates those parameters using objective estimates from repeated circuit measurements. This notebook keeps the QUBO construction and schematic circuit-design stages visible, but its classical surrogate sampler is not a QAOA measurement model. Consequently, its parameter search and candidate scores are demonstrations of software flow, not evidence about QAOA performance.

### What to inspect

Examine the variable count, matrix structure, circuit description, sampled candidates, and explicit constraint checks. Treat reported costs as outputs of this simplified model, not as operationally validated IRP plans or a comparison establishing quantum advantage.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple
import itertools
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class QUBOVariable:
    """A binary decision or slack variable in the QUBO model."""

    name: str
    index: int
    variable_type: str
    domain: Tuple[int, ...]


@dataclass
class QUBOTerm:
    """A linear or quadratic contribution to the QUBO energy."""

    coefficients: List[float]
    variables: List[QUBOVariable]
    term_type: str
    description: str


@dataclass
class QuantumCircuit:
    """Schematic QAOA circuit description; not an executable circuit."""

    num_qubits: int
    depth: int
    parameters: List[float]
    gates: List[Dict[str, Any]]
    measurement_basis: str


@dataclass
class QuantumSolution:
    """A sampled bit string and its decoded model decisions."""

    binary_string: str
    routes: Dict[Tuple[int, int], List[int]]
    deliveries: Dict[Tuple[int, int], int]
    objective_value: float
    constraint_violations: List[str]
    surrogate_score: float
    classical_refinement: Dict[str, Any]


class QuantumIRP:
    """Build a penalized QUBO and a schematic QAOA circuit for a toy IRP."""

    def __init__(self, depot, customers, vehicles, num_periods):
        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.qaoa_depth = 3
        self.penalty_weight = 100.0
        self.qubo_variables = []
        self.qubo_terms = []
        self.qubo_matrix = None
        self.quantum_circuit = None
        self.quantum_solutions = []

        print(
            f"Initialized the model with {len(customers)} customers, "
            f"{len(vehicles)} vehicles, and {num_periods} periods."
        )
        print(
            f"Schematic QAOA depth: {self.qaoa_depth}; "
            f"binary variables: {self._estimate_qubits_needed()}"
        )

    def _estimate_qubits_needed(self) -> int:
        """Count decision and slack bits required by this encoding."""
        vehicle_count = len(self.vehicles)
        customer_count = len(self.customers)
        periods = self.num_periods
        route_bits = vehicle_count * customer_count * periods
        delivery_bits = customer_count * periods
        inventory_bits = customer_count * periods
        activation_bits = vehicle_count * periods
        capacity_slack_bits = 2 * vehicle_count * periods
        return (
            route_bits
            + delivery_bits
            + inventory_bits
            + activation_bits
            + capacity_slack_bits
        )

    def formulate_qubo(self) -> None:
        """Create variables, operating-cost terms, penalties, and Q."""
        print("\n=== QUBO FORMULATION ===")
        self.qubo_variables = []
        self.qubo_terms = []
        self._initialize_qubo_variables()
        self._add_objective_terms()
        self._add_constraint_terms()
        self._build_qubo_matrix()
        print(f"Variables: {len(self.qubo_variables)}")
        print(f"Recorded terms: {len(self.qubo_terms)}")
        print(f"QUBO matrix: {self.qubo_matrix.shape[0]} x {self.qubo_matrix.shape[1]}")

    def _initialize_qubo_variables(self) -> None:
        """Assign a stable bit index to every model variable."""
        self.variable_lookup = {}

        def add_variable(name, variable_type, domain):
            variable = QUBOVariable(
                name=name,
                index=len(self.qubo_variables),
                variable_type=variable_type,
                domain=tuple(domain),
            )
            self.qubo_variables.append(variable)
            self.variable_lookup[(variable_type, tuple(domain))] = variable

        for v in range(len(self.vehicles)):
            for c in range(len(self.customers)):
                for t in range(self.num_periods):
                    add_variable(f"x_{v}_{c}_{t}", "route", (v, c, t))

        for c in range(len(self.customers)):
            for t in range(self.num_periods):
                add_variable(f"y_{c}_{t}", "delivery", (c, t))
                add_variable(f"z_{c}_{t}", "inventory", (c, t))

        for v in range(len(self.vehicles)):
            for t in range(self.num_periods):
                add_variable(f"a_{v}_{t}", "activation", (v, t))
                add_variable(f"s1_{v}_{t}", "capacity_slack", (v, t, 1))
                add_variable(f"s2_{v}_{t}", "capacity_slack", (v, t, 2))

    def _get_variable(self, var_type: str, domain: Tuple[int, ...]) -> QUBOVariable:
        """Retrieve a variable by its semantic type and index domain."""
        try:
            return self.variable_lookup[(var_type, tuple(domain))]
        except KeyError as exc:
            raise ValueError(f"Unknown QUBO variable: {var_type}, {domain}") from exc

    def _add_linear_term(self, variable, coefficient, description, term_type="linear"):
        self.qubo_terms.append(
            QUBOTerm([float(coefficient)], [variable], term_type, description)
        )

    def _add_quadratic_term(
        self, variable_i, variable_j, coefficient, description, term_type="quadratic"
    ):
        self.qubo_terms.append(
            QUBOTerm(
                [float(coefficient)],
                [variable_i, variable_j],
                term_type,
                description,
            )
        )

    def _add_squared_penalty(self, expression, rhs, description):
        """Expand penalty_weight * (sum(a_i*x_i) - rhs)^2 for binary bits."""
        items = list(expression.items())
        constant = -float(rhs)
        weight = self.penalty_weight

        for variable, coefficient in items:
            linear_coefficient = weight * (
                coefficient ** 2 + 2.0 * constant * coefficient
            )
            self._add_linear_term(
                variable,
                linear_coefficient,
                description,
                term_type="constraint_penalty",
            )

        for (variable_i, coefficient_i), (variable_j, coefficient_j) in itertools.combinations(
            items, 2
        ):
            self._add_quadratic_term(
                variable_i,
                variable_j,
                2.0 * weight * coefficient_i * coefficient_j,
                description,
                term_type="constraint_penalty",
            )

    def _add_objective_terms(self) -> None:
        """Add the distance, vehicle-use, delivery, and holding costs."""
        print("Adding operating-cost terms...")
        for v, vehicle in enumerate(self.vehicles):
            for c, customer in enumerate(self.customers):
                distance = np.hypot(customer.x - self.depot.x, customer.y - self.depot.y)
                for t in range(self.num_periods):
                    route = self._get_variable("route", (v, c, t))
                    self._add_linear_term(
                        route,
                        distance * vehicle.variable_cost,
                        f"Travel cost for vehicle {v}, customer {c}, period {t}",
                    )

        for v, vehicle in enumerate(self.vehicles):
            for t in range(self.num_periods):
                activation = self._get_variable("activation", (v, t))
                self._add_linear_term(
                    activation,
                    vehicle.fixed_cost,
                    f"Fixed cost for vehicle {v}, period {t}",
                )

        for c, customer in enumerate(self.customers):
            for t in range(self.num_periods):
                delivery = self._get_variable("delivery", (c, t))
                inventory = self._get_variable("inventory", (c, t))
                self._add_linear_term(
                    delivery, 0.5, f"Binary delivery cost for customer {c}, period {t}"
                )
                self._add_linear_term(
                    inventory,
                    customer.holding_cost,
                    f"Inventory-proxy cost for customer {c}, period {t}",
                )

    def _add_constraint_terms(self) -> None:
        """Encode the stated visit, activation, delivery, and capacity constraints."""
        print("Adding constraint penalties...")

        for c in range(len(self.customers)):
            for t in range(self.num_periods):
                visit_variables = [
                    self._get_variable("route", (v, c, t))
                    for v in range(len(self.vehicles))
                ]
                for variable_i, variable_j in itertools.combinations(visit_variables, 2):
                    self._add_quadratic_term(
                        variable_i,
                        variable_j,
                        self.penalty_weight,
                        f"At-most-one-vehicle visit for customer {c}, period {t}",
                        term_type="constraint_penalty",
                    )

                delivery = self._get_variable("delivery", (c, t))
                delivery_expression = {delivery: 1.0}
                for route in visit_variables:
                    delivery_expression[route] = -1.0
                self._add_squared_penalty(
                    delivery_expression,
                    0.0,
                    f"Delivery equals visits for customer {c}, period {t}",
                )

                inventory = self._get_variable("inventory", (c, t))
                self._add_squared_penalty(
                    {inventory: 1.0, delivery: -1.0},
                    0.0,
                    f"Inventory proxy equals delivery for customer {c}, period {t}",
                )

        for v in range(len(self.vehicles)):
            for t in range(self.num_periods):
                activation = self._get_variable("activation", (v, t))
                for c in range(len(self.customers)):
                    route = self._get_variable("route", (v, c, t))
                    # This penalty is zero unless a route is selected while activation is zero.
                    self._add_linear_term(
                        route,
                        self.penalty_weight,
                        f"Vehicle activation link for vehicle {v}, customer {c}, period {t}",
                        term_type="constraint_penalty",
                    )
                    self._add_quadratic_term(
                        route,
                        activation,
                        -self.penalty_weight,
                        f"Vehicle activation link for vehicle {v}, customer {c}, period {t}",
                        term_type="constraint_penalty",
                    )

                slack_1 = self._get_variable("capacity_slack", (v, t, 1))
                slack_2 = self._get_variable("capacity_slack", (v, t, 2))
                capacity_expression = {slack_1: 1.0, slack_2: 2.0}
                for c in range(len(self.customers)):
                    route = self._get_variable("route", (v, c, t))
                    capacity_expression[route] = 1.0
                self._add_squared_penalty(
                    capacity_expression,
                    3.0,
                    f"Maximum of three visits for vehicle {v}, period {t}",
                )

    def _build_qubo_matrix(self) -> None:
        """Build the upper-triangular matrix used by the energy convention."""
        variable_count = len(self.qubo_variables)
        self.qubo_matrix = np.zeros((variable_count, variable_count), dtype=float)

        for term in self.qubo_terms:
            first = term.variables[0].index
            if len(term.variables) == 1:
                self.qubo_matrix[first, first] += term.coefficients[0]
            else:
                second = term.variables[1].index
                row, column = sorted((first, second))
                if row == column:
                    self.qubo_matrix[row, row] += term.coefficients[0]
                else:
                    self.qubo_matrix[row, column] += term.coefficients[0]

        print(
            f"Nonzero QUBO coefficients: {np.count_nonzero(self.qubo_matrix)}; "
            f"density: {np.count_nonzero(self.qubo_matrix) / self.qubo_matrix.size:.4f}"
        )

    def design_quantum_circuit(self) -> None:
        """Describe the alternating QAOA layers without executing them."""
        if self.qubo_matrix is None:
            raise RuntimeError("Formulate the QUBO before designing its circuit.")

        print("\n=== SCHEMATIC QAOA CIRCUIT DESIGN ===")
        qubit_count = len(self.qubo_variables)
        parameters = [
            float(np.random.uniform(0.0, 2.0 * np.pi))
            for _ in range(2 * self.qaoa_depth)
        ]
        gates = [
            {"type": "H", "target": index, "layer": "initial"}
            for index in range(qubit_count)
        ]

        for layer in range(self.qaoa_depth):
            gamma = parameters[2 * layer]
            beta = parameters[2 * layer + 1]
            for i in range(qubit_count):
                for j in range(i, qubit_count):
                    coefficient = self.qubo_matrix[i, j]
                    if coefficient != 0.0:
                        gates.append(
                            {
                                "type": "COST_TERM",
                                "targets": [i] if i == j else [i, j],
                                "coefficient": float(coefficient),
                                "gamma": gamma,
                                "layer": f"cost_{layer}",
                            }
                        )
            for index in range(qubit_count):
                gates.append(
                    {
                        "type": "MIXER_ROTATION",
                        "target": index,
                        "beta": beta,
                        "layer": f"mixer_{layer}",
                    }
                )

        gates.extend(
            {"type": "MEASURE", "target": index, "layer": "measurement"}
            for index in range(qubit_count)
        )
        self.quantum_circuit = QuantumCircuit(
            num_qubits=qubit_count,
            depth=self.qaoa_depth,
            parameters=parameters,
            gates=gates,
            measurement_basis="computational",
        )
        print(
            f"Schematic description: {qubit_count} qubits, "
            f"{self.qaoa_depth} layers, {len(gates)} gate records, "
            f"{len(parameters)} parameters."
        )

    def _calculate_energy(self, binary_string: str) -> float:
        """Evaluate E(q) using the upper-triangular QUBO convention."""
        if self.qubo_matrix is None:
            raise RuntimeError("Formulate the QUBO before evaluating energy.")
        if len(binary_string) != len(self.qubo_variables) or set(binary_string) - {"0", "1"}:
            raise ValueError("The bit string must contain one binary bit per QUBO variable.")

        bits = np.fromiter((int(bit) for bit in binary_string), dtype=float)
        energy = float(np.dot(np.diag(self.qubo_matrix), bits))
        for i in range(len(bits)):
            energy += float(np.dot(self.qubo_matrix[i, i + 1 :], bits[i + 1 :]) * bits[i])
        return energy

    def simulate_quantum_execution(self, params: List[float]) -> List[str]:
        """Generate classical surrogate samples, not QAOA circuit measurements."""
        if self.qubo_matrix is None:
            raise RuntimeError("Formulate the QUBO before generating samples.")
        if len(params) != 2 * self.qaoa_depth:
            raise ValueError("The parameter list must contain gamma and beta for each layer.")

        # Independent biased bits are deliberately a toy sampler: they do not reproduce
        # quantum interference, entanglement, or the distribution of a QAOA circuit.
        diagonal = np.diag(self.qubo_matrix)
        scale = max(1.0, float(np.max(np.abs(diagonal))))
        parameter_signal = float(np.mean(np.cos(params)))
        probabilities = 1.0 / (1.0 + np.exp(np.clip(diagonal / scale - parameter_signal, -30, 30)))
        probabilities = np.clip(probabilities, 0.05, 0.95)
        return [
            "".join("1" if random.random() < probability else "0" for probability in probabilities)
            for _ in range(100)
        ]

    def classical_parameter_optimization(self) -> List[float]:
        """Run a small grid search against the explicitly classical surrogate."""
        print("\n=== SURROGATE PARAMETER GRID SEARCH ===")
        best_parameters = None
        best_mean_energy = float("inf")

        for gamma in np.linspace(0.0, np.pi, 5):
            for beta in np.linspace(0.0, np.pi / 2.0, 5):
                parameters = [float(gamma), float(beta)] * self.qaoa_depth
                samples = self.simulate_quantum_execution(parameters)
                mean_energy = float(np.mean([self._calculate_energy(sample) for sample in samples]))
                if mean_energy < best_mean_energy:
                    best_mean_energy = mean_energy
                    best_parameters = parameters

        print(f"Best surrogate mean energy: {best_mean_energy:.2f}")
        print("This is not a QAOA expectation estimate.")
        return best_parameters

    def _binary_to_solution(self, binary_string):
        """Decode route and delivery bits using the variable index map."""
        if len(binary_string) != len(self.qubo_variables):
            raise ValueError("The sample length does not match the QUBO variable count.")
        routes = {}
        deliveries = {}
        for variable in self.qubo_variables:
            if binary_string[variable.index] != "1":
                continue
            if variable.variable_type == "route":
                v, c, t = variable.domain
                routes.setdefault((v, t), []).append(c)
            elif variable.variable_type == "delivery":
                c, t = variable.domain
                deliveries[(c, t)] = 1
        return routes, deliveries

    def _calculate_objective_value(self, routes, deliveries):
        """Calculate the operating cost, excluding constraint penalties."""
        total_cost = 0.0
        used_vehicle_periods = set()
        for (v, t), customer_indices in routes.items():
            if customer_indices:
                used_vehicle_periods.add((v, t))
            for c in customer_indices:
                customer = self.customers[c]
                vehicle = self.vehicles[v]
                distance = np.hypot(customer.x - self.depot.x, customer.y - self.depot.y)
                total_cost += distance * vehicle.variable_cost
        for v, _ in used_vehicle_periods:
            total_cost += self.vehicles[v].fixed_cost
        total_cost += 0.5 * sum(deliveries.values())
        total_cost += sum(
            self.customers[c].holding_cost
            for c, t in deliveries
        )
        return total_cost

    def _check_constraints(self, routes, deliveries):
        """Check visit uniqueness, vehicle capacity, and route-delivery consistency."""
        violations = []
        customer_period_visits = {}
        for (v, t), customer_indices in routes.items():
            if len(customer_indices) > 3:
                violations.append(f"Vehicle {v}, period {t}: more than three visits")
            if len(customer_indices) != len(set(customer_indices)):
                violations.append(f"Vehicle {v}, period {t}: duplicate customer visit")
            for c in customer_indices:
                customer_period_visits[(c, t)] = customer_period_visits.get((c, t), 0) + 1
        for (c, t), visit_count in customer_period_visits.items():
            if visit_count > 1:
                violations.append(f"Customer {c}, period {t}: visited by multiple vehicles")
        for customer_period, visit_count in customer_period_visits.items():
            if visit_count != deliveries.get(customer_period, 0):
                violations.append(f"Customer {customer_period}: delivery and visit decisions differ")
        for customer_period, delivery in deliveries.items():
            if delivery and customer_period_visits.get(customer_period, 0) != 1:
                violations.append(f"Customer {customer_period}: delivery has no unique visit")
        return violations

    def _classical_refinement(self, routes, deliveries):
        """Return a transparent duplicate-removal and consistency post-processing record."""
        refined_routes = {
            key: sorted(set(customer_indices))
            for key, customer_indices in routes.items()
        }
        visited = {
            (c, t)
            for (v, t), customer_indices in refined_routes.items()
            for c in customer_indices
        }
        refined_deliveries = {
            customer_period: 1
            for customer_period in deliveries
            if customer_period in visited
        }
        return {
            "refined_routes": refined_routes,
            "refined_deliveries": refined_deliveries,
            "description": "Duplicate removal and route-delivery consistency post-processing",
        }

    def extract_solutions(self, samples):
        """Decode, evaluate, and rank surrogate samples by feasibility and cost."""
        print("\n=== SAMPLE DECODING AND FEASIBILITY CHECK ===")
        solutions = []
        for sample in samples:
            routes, deliveries = self._binary_to_solution(sample)
            solutions.append(
                QuantumSolution(
                    binary_string=sample,
                    routes=routes,
                    deliveries=deliveries,
                    objective_value=self._calculate_objective_value(routes, deliveries),
                    constraint_violations=self._check_constraints(routes, deliveries),
                    surrogate_score=self._calculate_energy(sample),
                    classical_refinement=self._classical_refinement(routes, deliveries),
                )
            )
        solutions.sort(
            key=lambda solution: (
                bool(solution.constraint_violations),
                solution.objective_value,
                solution.surrogate_score,
            )
        )
        feasible_count = sum(not solution.constraint_violations for solution in solutions)
        print(f"Decoded samples: {len(solutions)}")
        print(f"Samples passing the stated checks: {feasible_count}")
        return solutions

    def _run_classical_baseline(self):
        """Construct a simple deterministic baseline for context, not an optimizer."""
        routes = {}
        deliveries = {}
        vehicle_count = len(self.vehicles)
        for c in range(len(self.customers)):
            v = c % vehicle_count
            t = (c // vehicle_count) // 3
            if t < self.num_periods:
                routes.setdefault((v, t), []).append(c)
                deliveries[(c, t)] = 1
        return {
            "routes": routes,
            "deliveries": deliveries,
            "objective": self._calculate_objective_value(routes, deliveries),
            "violations": self._check_constraints(routes, deliveries),
            "method": "deterministic assignment baseline; not an optimization solver",
        }

    def analyze_scaling_behavior(self, problem_sizes):
        """Report encoding-size counts without asserting runtime or quantum speedup."""
        results = {}
        vehicle_count = len(self.vehicles)
        periods = self.num_periods
        for size in problem_sizes:
            results[size] = {
                "route_bits": vehicle_count * size * periods,
                "delivery_bits": size * periods,
                "inventory_bits": size * periods,
                "activation_bits": vehicle_count * periods,
                "capacity_slack_bits": 2 * vehicle_count * periods,
                "total_bits": size * periods * (vehicle_count + 2)
                + 3 * vehicle_count * periods,
            }
        return results

    def run_quantum_optimization(self):
        """Run QUBO construction and the clearly labeled surrogate workflow."""
        start_time = time.time()
        self.formulate_qubo()
        self.design_quantum_circuit()
        parameters = self.classical_parameter_optimization()
        samples = self.simulate_quantum_execution(parameters)
        self.quantum_solutions = self.extract_solutions(samples)
        baseline = self._run_classical_baseline()
        elapsed_time = time.time() - start_time

        best_feasible = next(
            (solution for solution in self.quantum_solutions
             if not solution.constraint_violations),
            None,
        )
        results = {
            "quantum_solutions": self.quantum_solutions,
            "best_feasible_solution": best_feasible,
            "classical_solution": baseline,
            "optimization_time": elapsed_time,
            "qubo_formulation": {
                "variables": len(self.qubo_variables),
                "terms": len(self.qubo_terms),
                "matrix_size": len(self.qubo_matrix),
                "nonzero_coefficients": int(np.count_nonzero(self.qubo_matrix)),
            },
            "quantum_circuit": {
                "qubits": self.quantum_circuit.num_qubits,
                "depth": self.quantum_circuit.depth,
                "gates": len(self.quantum_circuit.gates),
                "parameters": len(self.quantum_circuit.parameters),
            },
            "scaling_counts": self.analyze_scaling_behavior([8, 12, 16, 20, 24]),
            "interpretation": (
                "Classical surrogate samples and a schematic circuit; "
                "no quantum execution or quantum-advantage measurement."
            ),
        }
        return results

In [ ]:
def create_quantum_instance():
    """Create the fixed, seeded example instance used in this notebook."""
    depot = type(
        "Depot",
        (),
        {"x": 0.0, "y": 0.0, "initial_inventory": 800.0, "holding_cost": 0.1},
    )()

    np.random.seed(42)
    random.seed(42)
    customers = []
    for customer_index in range(8):
        cluster_x = np.random.choice([-10, 0, 10])
        cluster_y = np.random.choice([-10, 0, 10])
        customer = type(
            "Customer",
            (),
            {
                "id": customer_index + 1,
                "x": cluster_x + np.random.normal(0, 4),
                "y": cluster_y + np.random.normal(0, 4),
                "demand_mean": np.random.uniform(15, 35),
                "demand_std": np.random.uniform(4, 8),
                "holding_cost": np.random.uniform(0.1, 0.3),
                "initial_inventory": np.random.uniform(40, 80),
                "min_inventory": np.random.uniform(15, 25),
                "max_inventory": np.random.uniform(90, 130),
            },
        )()
        customers.append(customer)

    vehicles = [
        type(
            "Vehicle",
            (),
            {"id": 1, "capacity": 150, "fixed_cost": 180, "variable_cost": 2.5},
        )(),
        type(
            "Vehicle",
            (),
            {"id": 2, "capacity": 150, "fixed_cost": 180, "variable_cost": 2.5},
        )(),
    ]
    return depot, customers, vehicles


print("Creating the seeded IRP example instance...")
depot, customers, vehicles = create_quantum_instance()
print(f"Depot location: ({depot.x:.1f}, {depot.y:.1f})")
print("Customer data (coordinates, mean demand, initial inventory):")
for customer in customers:
    print(
        f"  Customer {customer.id}: ({customer.x:.1f}, {customer.y:.1f}), "
        f"demand {customer.demand_mean:.1f} +/- {customer.demand_std:.1f}, "
        f"inventory {customer.initial_inventory:.1f}"
    )
print("Vehicle data (capacity, fixed cost, variable cost):")
for vehicle in vehicles:
    print(
        f"  Vehicle {vehicle.id}: capacity {vehicle.capacity}, "
        f"fixed cost {vehicle.fixed_cost}, variable cost {vehicle.variable_cost}"
    )

In [ ]:
quantum_irp = QuantumIRP(depot, customers, vehicles, num_periods=4)
results = quantum_irp.run_quantum_optimization()

print("\n=== WORKFLOW SUMMARY ===")
print(f"QUBO variables: {results['qubo_formulation']['variables']}")
print(f"Nonzero QUBO coefficients: {results['qubo_formulation']['nonzero_coefficients']}")
print(f"Schematic circuit depth: {results['quantum_circuit']['depth']}")
print(f"Surrogate samples decoded: {len(results['quantum_solutions'])}")
print(
    "Best sampled feasible operating cost: "
    f"{results['best_feasible_solution'].objective_value:.2f}"
    if results["best_feasible_solution"] is not None
    else "No sampled candidate passed all stated feasibility checks."
)
print(f"Baseline operating cost: {results['classical_solution']['objective']:.2f}")
print(f"Elapsed workflow time: {results['optimization_time']:.2f} seconds")
print(results["interpretation"])

assert results["qubo_formulation"]["variables"] == quantum_irp._estimate_qubits_needed()
assert quantum_irp.qubo_matrix.shape == (
    len(quantum_irp.qubo_variables),
    len(quantum_irp.qubo_variables),
)
assert np.allclose(np.tril(quantum_irp.qubo_matrix, k=-1), 0.0)
assert len(results["quantum_solutions"]) == 100

In [ ]:
def analyze_quantum_performance(quantum_irp, results):
    """Summarize encoding, circuit description, feasibility, and scaling counts."""
    qubo_info = results["qubo_formulation"]
    circuit_info = results["quantum_circuit"]
    feasible = [
        solution
        for solution in results["quantum_solutions"]
        if not solution.constraint_violations
    ]

    print("=== MODEL AND SAMPLING ANALYSIS ===")
    print(f"Binary variables: {qubo_info['variables']}")
    print(f"Recorded QUBO terms: {qubo_info['terms']}")
    print(f"Nonzero matrix coefficients: {qubo_info['nonzero_coefficients']}")
    print(
        f"Matrix density: {qubo_info['nonzero_coefficients'] / (qubo_info['matrix_size'] ** 2):.4f}"
    )
    print(
        f"Schematic circuit: {circuit_info['qubits']} qubits, "
        f"depth {circuit_info['depth']}, {circuit_info['gates']} gate records, "
        f"{circuit_info['parameters']} parameters"
    )
    print(f"Surrogate samples passing checks: {len(feasible)} of {len(results['quantum_solutions'])}")

    if feasible:
        best = min(feasible, key=lambda solution: solution.objective_value)
        print(f"Lowest feasible sampled operating cost: {best.objective_value:.2f}")
        print(f"Its QUBO energy, including penalties: {best.surrogate_score:.2f}")
    else:
        best = None
        print("No sampled candidate passed all stated feasibility checks.")

    print(f"Deterministic baseline operating cost: {results['classical_solution']['objective']:.2f}")
    print("The baseline is a reference assignment, not an optimality certificate.")
    print("No quantum execution time, speedup, or advantage is measured.")

    return {
        "feasible_sample_count": len(feasible),
        "best_feasible_cost": best.objective_value if best is not None else None,
        "scaling_counts": results["scaling_counts"],
    }


quantum_analysis = analyze_quantum_performance(quantum_irp, results)

In [ ]:
def visualize_quantum_results(quantum_irp, results, quantum_analysis):
    """Plot the QUBO coefficients, sampled costs, and encoding-size counts."""
    figure, axes = plt.subplots(1, 3, figsize=(17, 5))
    figure.suptitle("QUBO Model and Classical Surrogate Diagnostics", fontsize=14)

    matrix = quantum_irp.qubo_matrix
    shown_size = min(30, len(matrix))
    image = axes[0].imshow(
        matrix[:shown_size, :shown_size],
        cmap="coolwarm",
        aspect="auto",
    )
    axes[0].set_title(f"Upper-triangular QUBO coefficients (first {shown_size})")
    axes[0].set_xlabel("Variable index")
    axes[0].set_ylabel("Variable index")
    colorbar = figure.colorbar(image, ax=axes[0])
    colorbar.set_label("Coefficient value")

    feasible_costs = [
        solution.objective_value
        for solution in results["quantum_solutions"]
        if not solution.constraint_violations
    ]
    infeasible_costs = [
        solution.objective_value
        for solution in results["quantum_solutions"]
        if solution.constraint_violations
    ]
    if feasible_costs:
        axes[1].hist(feasible_costs, bins=12, alpha=0.75, label="Feasible surrogate samples")
    if infeasible_costs:
        axes[1].hist(infeasible_costs, bins=12, alpha=0.55, label="Infeasible surrogate samples")
    axes[1].set_title("Operating costs of decoded samples")
    axes[1].set_xlabel("Operating cost (model units)")
    axes[1].set_ylabel("Sample count")
    axes[1].legend()
    axes[1].grid(True, alpha=0.25)

    sizes = list(quantum_analysis["scaling_counts"])
    bit_counts = [quantum_analysis["scaling_counts"][size]["total_bits"] for size in sizes]
    axes[2].plot(sizes, bit_counts, marker="o", label="Binary variables in this encoding")
    axes[2].set_title("Encoding size as customer count changes")
    axes[2].set_xlabel("Number of customers")
    axes[2].set_ylabel("Binary variable count")
    axes[2].legend()
    axes[2].grid(True, alpha=0.25)

    figure.tight_layout()
    plt.show()


visualize_quantum_results(quantum_irp, results, quantum_analysis)

## Interpreting the results

The QUBO matrix represents the operating-cost terms and the selected constraint penalties. Its variable count is determined by the encoding: route, delivery, inventory-proxy, vehicle-activation, and capacity-slack bits. The coefficient plot is a view of that matrix, not a performance chart.

The sample histogram summarizes the output of the notebook's independent-bit classical surrogate. Feasibility is assessed separately from operating cost because a low operating cost alone does not imply that the route and delivery decisions satisfy the model constraints. The reported deterministic baseline is a reference assignment, not an exact solution or a competing optimizer.

The scaling plot counts binary variables as the number of customers changes while the vehicle count and planning horizon remain fixed. It does not estimate runtime, circuit difficulty, hardware feasibility, or quantum speedup. In particular, a larger theoretical solution space does not by itself imply a quantum advantage.

### Modeling and implementation limits

- Inventory is a binary proxy tied to same-period delivery, not an inventory trajectory. Demand, replenishment quantities, inventory balance, depot stock, and stockout penalties are outside this formulation.
- Travel costs use depot-to-customer distance independently for each visit. Route sequencing, inter-customer travel, time windows, and vehicle load quantities are not represented.
- The finite penalty weight influences the trade-off between operating cost and constraint satisfaction. Feasibility checks remain necessary when interpreting sampled bit strings.
- The displayed circuit is schematic, and the sampling and parameter-search procedures are classical teaching surrogates. They do not model QAOA's quantum state, measurement distribution, hardware noise, or a real hybrid quantum-classical optimization loop.
- Accordingly, this notebook demonstrates a QUBO modeling workflow and the structure of a QAOA-inspired circuit description. It makes no empirical claim of quantum speedup or quantum advantage.

A fuller IRP study would require a richer inventory-flow model and an executable circuit or simulator that evaluates the corresponding cost Hamiltonian. Those extensions change the modeling and computational scope and are not implemented here.